# Adiabaticity: in-situ training profiles vs synthetic profiles

Compares the Wood (2005) adiabaticity of the in-situ profiles that train the synthetic-profile generator (`06_synthetic_profile_generator.py`) with the adiabaticity of the profiles it generates.

**Adiabaticity.** Wood (2005, J. Atmos. Sci. 62, 3011–3033, sec. 3b):

$$
f_{ad} = \frac{LWP_{obs}}{LWP_{ad}}, \qquad LWP_{ad} = \tfrac{1}{2}\,\Gamma_{ad}\,h^2
$$

where h is the cloud thickness from the cloud base z_cb to the cloud top, and Γ_ad is the adiabatic rate of increase of LWC with height, evaluated once per profile at the mean in-cloud temperature and pressure of the ERA5 column.

Every f_ad here comes from `plot_per_cloud_rmse_kfold.wood_adiabaticity`, the same function the k-fold figures use and the Python twin of the MATLAB `compute_adiabaticity_Wood2005.m`. So the in-situ and synthetic numbers are computed identically:
- **Cloud base:** z-intercept of a line fit to LWC(z) over the bottom third of the profile, never above the lowest level.
- **Fallback:** if the fitted slope is ≤ 0.25 Γ_ad or > 3 Γ_ad, the base goes one adiabatic distance (0.05 g m⁻³ / Γ_ad) below the lowest level.
- **LWP_obs:** includes the unsampled wedge between the base and the lowest level.

**Three populations**

| | What it is | Why it is here |
|---|---|---|
| in-situ, native | the training profiles, every measured sample | what the generator learns from |
| in-situ, 7 levels | the same profiles resampled to 7 evenly spaced levels, as the generator outputs them | removes the vertical-resolution difference: a synthetic bottom-third fit has only 3 points |
| synthetic | the generator's draws | what the network trains on |

Native vs 7 levels isolates the effect of vertical resolution; 7 levels vs synthetic isolates the sampler.

**Which in-situ profiles.** The generator saves `train_source_index`, the rows of the input file it actually trained on, after its τ_c limits, `OUTLIERS` and de-duplication. This notebook uses exactly those rows, so the comparison is always against the real training set. For an older `.npz` without that field it re-applies the τ_c limits and de-duplication, cannot apply `OUTLIERS`, and says so.

**How to run.** Set `SYNTHETIC_NPZ` in the next cell, or leave it to pick the newest `synthetic_profiles/synthetic_profiles_jointMVN_N100000_*.npz`. `COMPARE_NPZ` optionally adds a second synthetic set (for example one made with `OUTLIERS = ()`) to panel (a) and the tables. Environment variables of the same names, prefixed `FAD_`, override both.

In [1]:
# ── Settings ───────────────────────────────────────────────────────────────────
import os
import glob
from pathlib import Path

REPO = Path('/Users/andrewbuggee/Documents/VS_CODE/Python-Research/lasp-CU-paper-3')

# Synthetic profiles to analyse. Default: the newest 100,000-profile run in synthetic_profiles/
SYNTHETIC_NPZ = os.environ.get('FAD_SYNTHETIC_NPZ') or max(
    glob.glob(str(REPO / 'synthetic_profiles' / 'synthetic_profiles_jointMVN_N100000_*.npz')),
    key=os.path.getmtime)
SYNTHETIC_LABEL = os.environ.get('FAD_SYNTHETIC_LABEL', 'synthetic')

# Optional second synthetic set, e.g. the same generator with OUTLIERS = () ('' or unset: none)
COMPARE_NPZ = os.environ.get('FAD_COMPARE_NPZ') or None
COMPARE_LABEL = os.environ.get('FAD_COMPARE_LABEL', 'synthetic, comparison run')

# In-situ input file. None: use the input_path the generator saved in SYNTHETIC_NPZ
INSITU_MAT = os.environ.get('FAD_INSITU_MAT') or None

N_LEVELS = 7            # the generator's fixed output grid (N_FIXED_LEVELS)
SEED = 42               # bootstrap random seed, printed with the results
N_BOOT = 5000           # bootstrap resamples

FIG_PATH = os.environ.get('FAD_FIG_PATH') or str(
    REPO / 'synthetic_profiles' / (Path(SYNTHETIC_NPZ).stem + '_adiabaticity_vs_insitu.png'))

print('synthetic :', SYNTHETIC_NPZ)
print('compare   :', COMPARE_NPZ)
print('figure    :', FIG_PATH)

synthetic : /Users/andrewbuggee/Documents/VS_CODE/Python-Research/lasp-CU-paper-3/synthetic_profiles/synthetic_profiles_jointMVN_N100000_L7_2026-11-11.npz
compare   : None
figure    : /Users/andrewbuggee/Documents/VS_CODE/Python-Research/lasp-CU-paper-3/synthetic_profiles/synthetic_profiles_jointMVN_N100000_L7_2026-11-11_adiabaticity_vs_insitu.png


In [2]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys
import numpy as np
import pandas as pd
import scipy.io
import scipy.stats
import matplotlib.pyplot as plt

# The f_ad rule and its thermodynamics live in plot_per_cloud_rmse_kfold.py
sys.path.insert(0, str(REPO))
import plot_per_cloud_rmse_kfold as pk

## f_ad and the quantities it is built from

`f_ad_details` returns f_ad together with the pieces that set it, so differences in f_ad can be traced to thickness, LWP or Γ_ad. Γ_ad comes from the ERA5 temperature column on its fixed pressure levels: the column is integrated hypsometrically to heights (`era5_T_P_at_heights`), interpolated to each profile level, and T and p are averaged over the sampled cloud depth, as in `wood_adiabaticity`.

In [3]:
def f_ad_details(lwc_g_m3: np.ndarray, z_km: np.ndarray, T_K: np.ndarray, P_hPa: np.ndarray) -> tuple:
    '''f_ad and its ingredients for one profile, using the pipeline's own functions.

    lwc_g_m3, z_km : LWC [g/m^3] and altitude [km] of each level, cloud top -> base
    T_K, P_hPa     : ERA5 temperature [K] on its pressure levels [hPa], surface -> TOA

    Returns (f_ad [-], h [m], LWP of the sampled levels [g/m^2], Gamma_ad [g/m^3 per km],
             used_fallback [bool])
    '''
    z_m = np.asarray(z_km, float) * 1000.0
    lwc = np.asarray(lwc_g_m3, float)
    h_sampled_m = z_m.max() - z_m.min()

    # Mean in-cloud T and p of the ERA5 column, exactly as wood_adiabaticity forms them
    T_lev, P_lev = pk.era5_T_P_at_heights(T_K, P_hPa, z_km)
    T_mean_K = abs(np.trapezoid(T_lev, z_m)) / h_sampled_m
    P_mean_Pa = abs(np.trapezoid(P_lev, z_m)) / h_sampled_m
    gamma_g_m3_per_m = pk.gamma_ad_kg_per_m4(T_mean_K, P_mean_Pa) * 1000.0

    # Cloud base from the bottom-third line (or the fallback) and the resulting thickness
    z_cb_m, _, used_fallback = pk.extrapolated_cloud_base(lwc, z_m, gamma_g_m3_per_m)
    h_m = z_m.max() - z_cb_m

    f_ad = pk.wood_adiabaticity(lwc, z_km, T_K, P_hPa)
    lwp_sampled_g_m2 = abs(np.trapezoid(lwc, z_m))
    return f_ad, h_m, lwp_sampled_g_m2, gamma_g_m3_per_m * 1e3, used_fallback


COLUMNS = ['f_ad', 'h_m', 'lwp_g_m2', 'gamma_ad_g_m3_km', 'fallback']

## In-situ training profiles

The rows come from `train_source_index` in the synthetic file. Each profile is used twice: as measured, and resampled to 7 evenly spaced levels between its top and its lowest sample. The second matches the generator's output grid (`np.linspace(z_top, z_base, N_FIXED_LEVELS)`), linear in height.

In [4]:
syn_file = np.load(SYNTHETIC_NPZ)
mat_path = INSITU_MAT or (str(syn_file['input_path']) if 'input_path' in syn_file else None)
if mat_path is None:
    raise SystemExit('No in-situ input file: set INSITU_MAT (this .npz predates input_path).')
d = scipy.io.loadmat(mat_path, squeeze_me=True)
P_insitu_hPa = np.asarray(d['era5_pressure_hPa'][0], float)
campaign_all = np.asarray([str(c) for c in d['campaign_each']])

if 'train_source_index' in syn_file:
    rows = np.asarray(syn_file['train_source_index'], int)
    print(f'Training rows from the generator: {len(rows)}; OUTLIERS excluded: '
          f'{list(syn_file["outliers"]) if "outliers" in syn_file else "none recorded"}')
else:
    # Older output: rebuild the training set from the filters the generator applies
    tau_c = np.asarray(d['tau_c'], float)
    lo, hi = float(syn_file['TAU_C_MIN']), float(syn_file['TAU_C_MAX']) if 'TAU_C_MAX' in syn_file else np.inf
    rows, seen = [], set()
    for i in range(len(tau_c)):
        fp = tuple(np.round(np.atleast_1d(np.asarray(d['re'][i], float))[:5], 4))
        if lo <= tau_c[i] <= hi and fp not in seen:
            seen.add(fp)
            rows.append(i)
    rows = np.asarray(rows)
    print(f'WARNING: no train_source_index in this .npz; rebuilt {len(rows)} rows from the tau_c '
          f'limits and de-duplication. OUTLIERS could not be applied.')

native, seven, campaign = [], [], []
for i in rows:
    z_km = np.atleast_1d(np.asarray(d['z'][i], float))          # top -> base
    lwc = np.atleast_1d(np.asarray(d['lwc'][i], float))
    T_K = np.asarray(d['era5_T_K'][i], float)
    native.append(f_ad_details(lwc, z_km, T_K, P_insitu_hPa))

    # Resample to the generator's 7 evenly spaced levels, top -> base
    z7_km = np.linspace(z_km[0], z_km[-1], N_LEVELS)
    order = np.argsort(z_km)
    lwc7 = np.interp(z7_km, z_km[order], lwc[order])
    seven.append(f_ad_details(lwc7, z7_km, T_K, P_insitu_hPa))
    campaign.append(campaign_all[i])

insitu_native = pd.DataFrame(native, columns=COLUMNS)
insitu_seven = pd.DataFrame(seven, columns=COLUMNS)
campaign = np.asarray(campaign)
print(f'In-situ profiles: {len(rows)} (ORACLES {np.sum(campaign == "oracles")}, '
      f'VOCALS-REx {np.sum(campaign == "vocals-rex")})')

Training rows from the generator: 290; OUTLIERS excluded: [np.str_('oracles 2017-08-30 111'), np.str_('oracles 2017-08-30 112')]
In-situ profiles: 290 (ORACLES 209, VOCALS-REx 81)


## Synthetic profiles

Arrays are loaded from the `.npz` once: indexing an `NpzFile` re-reads the whole array from disk every time, which makes a 100,000-profile loop take many minutes.

In [5]:
def synthetic_f_ad(npz_path: str) -> pd.DataFrame:
    '''f_ad and its ingredients for every profile in a generator output file.'''
    f = np.load(npz_path)
    lwc = f['lwc'].astype(float)                # (N, 7) g/m^3, top -> base
    z_km = f['z'].astype(float)                 # (N, 7) km
    T_K = f['T'].astype(float)                  # (N, 37) K, surface -> TOA
    P_hPa = np.asarray(f['era5_pressure'], float)
    return pd.DataFrame([f_ad_details(lwc[j], z_km[j], T_K[j], P_hPa) for j in range(len(lwc))],
                        columns=COLUMNS)


populations = {'in-situ, native': insitu_native, 'in-situ, 7 levels': insitu_seven,
               SYNTHETIC_LABEL: synthetic_f_ad(SYNTHETIC_NPZ)}
if COMPARE_NPZ:
    populations[COMPARE_LABEL] = synthetic_f_ad(COMPARE_NPZ)
for name, df in populations.items():
    print(f'{name:28s} n = {len(df)}')

in-situ, native              n = 290
in-situ, 7 levels            n = 290
synthetic                    n = 100000


## Summary statistics

Medians and percentiles of f_ad and of the quantities that set it, the fraction of profiles above adiabatic, and how often the cloud base came from the fallback.

The last columns measure how tightly LWP follows thickness. They give the correlation of log LWP with log h, and the slope of log LWP regressed on log h; a constant f_ad would give a slope of 2. A looser LWP–h relation spreads f_ad = LWP/(½ Γ_ad h²) even when LWP and h each have the right distribution.

In [6]:
def describe(df: pd.DataFrame) -> dict:
    ok = np.isfinite(df['f_ad'])
    x = df.loc[ok, 'f_ad']
    log_h, log_lwp = np.log(df.loc[ok, 'h_m']), np.log(df.loc[ok, 'lwp_g_m2'])
    return {'n': int(ok.sum()),
            'f_ad median': x.median(), 'p10': x.quantile(0.1), 'p25': x.quantile(0.25),
            'p75': x.quantile(0.75), 'p90': x.quantile(0.9), '% above 1': 100 * (x > 1).mean(),
            'h median (m)': df['h_m'].median(), 'LWP median (g/m2)': df['lwp_g_m2'].median(),
            'Gamma_ad median (g/m3/km)': df['gamma_ad_g_m3_km'].median(),
            '% fallback': 100 * df['fallback'].mean(),
            'corr(log h, log LWP)': np.corrcoef(log_h, log_lwp)[0, 1],
            'slope log LWP on log h': np.polyfit(log_h, log_lwp, 1)[0]}


summary = pd.DataFrame({name: describe(df) for name, df in populations.items()}).T
pd.options.display.float_format = '{:.3f}'.format
summary

,n,f_ad median,p10,p25,p75,p90,% above 1,h median (m),LWP median (g/m2),Gamma_ad median (g/m3/km),% fallback,"corr(log h, log LWP)",slope log LWP on log h
"in-situ, native",290.000,0.673,0.348,0.485,0.863,1.129,15.517,238.722,40.903,2.370,13.448,0.818,1.375
"in-situ, 7 levels",290.000,0.632,0.321,0.466,0.836,1.061,14.828,237.872,39.633,2.370,8.621,0.804,1.331
synthetic,100000.000,0.642,0.343,0.457,0.907,1.244,19.465,238.652,40.607,2.289,12.995,0.766,1.240


## Are the distributions different?

- **Two-sample Kolmogorov–Smirnov tests on f_ad.** These compare whole distributions, so they are sensitive to differences in spread, not just the median.
- **Bootstrap 95% confidence interval on the median difference, synthetic minus in-situ (7 levels).** The in-situ profiles are resampled with the seed below. The synthetic median is treated as exact, because with 100,000 profiles its sampling error is negligible next to that of about 290 in-situ profiles.

In [7]:
def ks(a: str, b: str) -> tuple:
    x, y = populations[a]['f_ad'].dropna(), populations[b]['f_ad'].dropna()
    r = scipy.stats.ks_2samp(x, y)
    return a, b, r.statistic, r.pvalue


names = list(populations)
pairs = [(names[0], names[1])] + [(names[1], n) for n in names[2:]] + [(names[0], n) for n in names[2:]]
print(pd.DataFrame([ks(a, b) for a, b in pairs], columns=['A', 'B', 'KS D', 'p']).to_string(index=False))

rng = np.random.default_rng(SEED)
x = populations['in-situ, 7 levels']['f_ad'].dropna().to_numpy()
boot_medians = np.median(x[rng.integers(0, len(x), (N_BOOT, len(x)))], axis=1)   # (N_BOOT,)
print(f'\nBootstrap: seed {SEED}, {N_BOOT} resamples of the {len(x)} in-situ profiles (7 levels)')
for n in names[2:]:
    m = populations[n]['f_ad'].median()
    lo, hi = m - np.percentile(boot_medians, [97.5, 2.5])
    print(f'  median f_ad, {n} - in-situ (7 levels): {m - np.median(x):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')

                A                 B  KS D     p
  in-situ, native in-situ, 7 levels 0.076 0.375
in-situ, 7 levels         synthetic 0.073 0.084
  in-situ, native         synthetic 0.072 0.092

Bootstrap: seed 42, 5000 resamples of the 290 in-situ profiles (7 levels)
  median f_ad, synthetic - in-situ (7 levels): +0.010  95% CI [-0.028, +0.045]


## Figure

- **(a)** Cumulative distribution of f_ad for each population. The vertical line marks adiabatic (f_ad = 1).
- **(b)** LWP against cloud thickness. Gray shading is the synthetic density; points are the native in-situ profiles. Dotted lines mark constant f_ad at the median Γ_ad. Panel (a) is these points projected across those lines.
- **(c, d)** Cumulative distributions of thickness and LWP on their own. If these match while (a) does not, the sampler gets each quantity right but pairs them too loosely.

In [8]:
# One color per population; the 7-level in-situ curve is dashed because it is the same
# profiles as the native one. Colors are the first three slots of the dataviz reference
# palette, checked for color-vision-deficiency separation.
colors = {'in-situ, native': '#2a78d6', 'in-situ, 7 levels': '#1baf7a', SYNTHETIC_LABEL: '#eb6834'}
styles = {'in-situ, 7 levels': '--'}
if COMPARE_NPZ:
    colors[COMPARE_LABEL] = '#eb6834'
    styles[COMPARE_LABEL] = ':'

plt.rcParams.update({'font.size': 10})
fig, axs = plt.subplots(2, 2, figsize=(12, 9.5))


def ecdf(ax, column: str, xlabel: str, title: str, logx: bool = False, xlim=None) -> None:
    for name, df in populations.items():
        x = np.sort(df[column].dropna().to_numpy())
        ax.step(x, np.arange(1, len(x) + 1) / len(x), where='post', color=colors[name],
                ls=styles.get(name, '-'), lw=2, label=f'{name} (n={len(x)}): median {np.median(x):.3g}')
    if logx:
        ax.set_xscale('log')
    if xlim:
        ax.set_xlim(xlim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('fraction of profiles at or below')
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.15)
    ax.legend(frameon=False, fontsize=8.5, loc='lower right')


ecdf(axs[0, 0], 'f_ad', r'$f_{ad} = LWP_{obs} / LWP_{ad}$', '(a) Adiabaticity', xlim=(0, 2))
axs[0, 0].axvline(1, color='0.55', lw=1)

# (b) LWP_obs = f_ad * (1/2) Gamma_ad h^2, so the y axis includes the unsampled wedge
ax = axs[0, 1]
syn = populations[SYNTHETIC_LABEL]
lwp_obs = lambda df: df['f_ad'] * 0.5 * df['gamma_ad_g_m3_km'] * 1e-3 * df['h_m'] ** 2
hb = ax.hexbin(syn['h_m'], lwp_obs(syn), gridsize=60, xscale='log', yscale='log', cmap='Greys',
               mincnt=1, bins='log')
ax.scatter(insitu_native['h_m'], lwp_obs(insitu_native), s=14, color=colors['in-situ, native'],
           edgecolor='w', lw=0.4, label='in-situ, native', zorder=3)
gamma_g_m3_m = np.nanmedian(np.concatenate([insitu_native['gamma_ad_g_m3_km'], syn['gamma_ad_g_m3_km']])) * 1e-3
h_line_m = np.logspace(np.log10(40), np.log10(1200), 50)
for f_line in [0.25, 0.5, 1.0]:
    ax.plot(h_line_m, f_line * 0.5 * gamma_g_m3_m * h_line_m ** 2, color='0.35', lw=1, ls=':')
    ax.text(h_line_m[-1], f_line * 0.5 * gamma_g_m3_m * h_line_m[-1] ** 2, f' $f_{{ad}}$={f_line:g}',
            fontsize=8, color='0.35', va='center')
ax.set_xlabel('cloud thickness h, from the cloud base used (m)')
ax.set_ylabel(r'$LWP_{obs}$ (g m$^{-2}$)')
ax.set_title(f'(b) LWP against thickness: {SYNTHETIC_LABEL} density (gray), in-situ points', fontsize=10)
ax.grid(alpha=0.15, which='both')
ax.legend(frameon=False, fontsize=8.5, loc='upper left')
fig.colorbar(hb, ax=ax, label='synthetic profiles per cell')

ecdf(axs[1, 0], 'h_m', 'cloud thickness h (m)', '(c) Thickness', logx=True)
ecdf(axs[1, 1], 'lwp_g_m2', r'LWP of the sampled layer (g m$^{-2}$)', '(d) Liquid water path', logx=True)

lo, hi = float(syn_file['TAU_C_MIN']), float(syn_file['TAU_C_MAX'])
fig.suptitle(f'Wood (2005) adiabaticity: in-situ training profiles vs synthetic profiles '
             f'(${lo:g} \\leq \\tau_c \\leq {hi:g}$; {Path(SYNTHETIC_NPZ).name})', fontsize=11)
fig.tight_layout()
fig.savefig(FIG_PATH, dpi=150)
print('saved', FIG_PATH)
plt.show()

saved /Users/andrewbuggee/Documents/VS_CODE/Python-Research/lasp-CU-paper-3/synthetic_profiles/synthetic_profiles_jointMVN_N100000_L7_2026-11-11_adiabaticity_vs_insitu.png


/var/folders/6n/ncm6cm4d2dn0s707np340zk40000gq/T/ipykernel_47849/3081452438.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
